In [ ]:
import pandas as pd
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.preprocessing import StandardScaler
df = pd.read_csv("dormmatch_cleaned.csv")

In [ ]:
numeric_features = ["sleep_duration","cleanliness_level","noise_tolerance","study_hours",
                   "social_activity","introvert_extrovert","guest_frequency","music_frequency"]
categorical_features = ["sharing_preference","weekend_activity","food_habit","smoking_preference",
                       "ac_preference","preferred_room_type","preferred_environment","roommate_interaction"]
encoded = pd.get_dummies(df[numeric_features + categorical_features], columns=categorical_features)
X = StandardScaler().fit_transform(encoded)
similarity = cosine_similarity(X)

In [ ]:
def precision_at_k(recommended, relevant, k):
    recommended = recommended[:k]
    return len(set(recommended) & set(relevant)) / k if k else 0

def recall_at_k(recommended, relevant, k):
    recommended = recommended[:k]
    return len(set(recommended) & set(relevant)) / len(relevant) if relevant else 0

def dcg_at_k(recommended, relevant, k):
    return sum((1/np.log2(rank+2)) for rank, item in enumerate(recommended[:k]) if item in relevant)

def ndcg_at_k(recommended, relevant, k):
    ideal = dcg_at_k(list(relevant)[:k], relevant, k)
    return dcg_at_k(recommended, relevant, k) / ideal if ideal else 0

In [ ]:
def synthetic_relevance(student_idx, candidate_idx, threshold=0.72):
    return similarity[student_idx, candidate_idx] >= threshold

In [ ]:
evaluation_rows = []
k = 5
for student_idx in range(len(df)):
    scores = similarity[student_idx].copy()
    scores[student_idx] = -1
    recommended = np.argsort(scores)[::-1][:k].tolist()
    relevant = [j for j in range(len(df)) if j != student_idx and synthetic_relevance(student_idx,j)]
    evaluation_rows.append({
        "student_index":student_idx,
        "precision_at_5":precision_at_k(recommended,relevant,k),
        "recall_at_5":recall_at_k(recommended,relevant,k),
        "ndcg_at_5":ndcg_at_k(recommended,relevant,k)
    })
evaluation_df = pd.DataFrame(evaluation_rows)
evaluation_df.head()

In [ ]:
evaluation_df[["precision_at_5","recall_at_5","ndcg_at_5"]].mean()

In [ ]:
evaluation_df[["precision_at_5","recall_at_5","ndcg_at_5"]].describe()

In [ ]:
evaluation_df.to_csv("dormmatch_evaluation_metrics.csv", index=False)

In [ ]:
import matplotlib.pyplot as plt
metrics = evaluation_df[["precision_at_5","recall_at_5","ndcg_at_5"]].mean()
metrics.plot(kind="bar", figsize=(7,4), ylim=(0,1), title="Average Recommendation Metrics")
plt.ylabel("Score")
plt.tight_layout()
plt.show()